# 🧠 TRIBE v2 Reel Studio (Colab launcher)

Upload a reel → see the **predicted brain response** (Meta FAIR's TRIBE v2): a side-by-side brain movie, an interactive 3D cortex,
intensity over time per brain system, and the peak moments.

**One-time setup**
1. **Runtime → Change runtime type → GPU** (a T4 works; an A100/L4 is faster).
2. TRIBE v2 uses **LLaMA 3.2-3B**, which is gated: on Hugging Face, open https://huggingface.co/meta-llama/Llama-3.2-3B and click
   *Agree / request access* (usually approved within minutes).
3. Create a token at https://huggingface.co/settings/tokens (type *Read*) and add it in Colab: 🔑 **Secrets** (left sidebar) →
   name `HF_TOKEN`, paste the token, and enable *Notebook access*.
4. **Runtime → Run all**. The last cell prints a public `https://….gradio.live` link. Open it and upload your reel.

The first analysis downloads ~20 GB of models (TRIBE, V-JEPA2, LLaMA, Wav2Vec-BERT, WhisperX), which takes several minutes.
After that, a 30–60 s reel takes about 2–5 minutes on a T4.

<small>Predictions describe an *average* viewer's cortex, not real measurements. TRIBE v2 is CC BY-NC 4.0 (non-commercial only).</small>

In [ ]:
#@title 1 · Check GPU + install (≈3–5 min)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ No GPU: switch the runtime to GPU first!"
!pip -q install uv
!uv pip install --system -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git" "transformers>=4.46" "faster-whisper>=1.1" "ctranslate2>=4.5" gradio plotly

In [ ]:
#@title 2 · Hugging Face login (needed for gated LLaMA 3.2)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab secret HF_TOKEN not found:", e)
if not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")
from huggingface_hub import login, HfApi
login(token=os.environ["HF_TOKEN"])
try:
    HfApi().model_info("meta-llama/Llama-3.2-3B")
    print("✅ LLaMA 3.2-3B access OK")
except Exception as e:
    print("❌ No access to meta-llama/Llama-3.2-3B yet. Request it on huggingface.co first.\n", e)

In [ ]:
%%writefile brain_viz.py
"""Visualization and summary helpers for TRIBE v2 predictions on fsaverage5.

Everything here is pure numpy / matplotlib / plotly, so it can be tested
without a GPU or the TRIBE weights.
"""

from __future__ import annotations

import functools
import logging

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib.collections import PolyCollection  # noqa: E402

logger = logging.getLogger(__name__)

N_VERTS_HEMI = 10242  # fsaverage5
VIEWS = {
    # name: (hemisphere, camera side along x: -1 = looking from the left)
    "Left lateral": ("left", -1),
    "Right lateral": ("right", 1),
    "Left medial": ("left", 1),
    "Right medial": ("right", -1),
}

# Destrieux (aparc.a2009s) parcels grouped into broad functional systems.
# These are anatomical approximations meant for a readable summary,
# not a substitute for functional localizers.
NETWORKS: dict[str, list[str]] = {
    "Visual": [
        "G_cuneus", "G_occipital_middle", "G_occipital_sup", "G_and_S_occipital_inf",
        "Pole_occipital", "S_calcarine", "G_oc-temp_med-Lingual",
        "S_oc_middle_and_Lunatus", "S_oc_sup_and_transversal", "S_occipital_ant",
        "S_parieto_occipital",
    ],
    "Faces & objects": [
        "G_oc-temp_lat-fusifor", "S_oc-temp_lat", "S_oc-temp_med_and_Lingual",
        "G_temporal_inf", "S_temporal_inf", "S_collat_transv_post",
    ],
    "Auditory": [
        "G_temp_sup-G_T_transv", "S_temporal_transverse", "G_temp_sup-Plan_tempo",
        "G_temp_sup-Plan_polar", "Lat_Fis-post",
    ],
    "Language": [
        "G_temp_sup-Lateral", "S_temporal_sup", "G_temporal_middle",
        "G_front_inf-Opercular", "G_front_inf-Triangul", "G_front_inf-Orbital",
        "Pole_temporal", "G_pariet_inf-Angular",
    ],
    "Attention": [
        "G_parietal_sup", "S_intrapariet_and_P_trans", "G_pariet_inf-Supramar",
        "S_precentral-sup-part",
    ],
    "Motor & touch": [
        "G_precentral", "G_postcentral", "S_central", "G_and_S_paracentral",
        "S_postcentral", "G_and_S_subcentral", "S_precentral-inf-part",
    ],
    "Prefrontal": [
        "G_front_sup", "G_front_middle", "S_front_sup", "S_front_middle",
        "S_front_inf", "G_and_S_frontomargin", "G_and_S_transv_frontopol",
    ],
    "Emotion & reward": [
        "G_orbital", "G_rectus", "S_orbital-H_Shaped", "S_orbital_lateral",
        "S_orbital_med-olfact", "S_suborbital", "G_subcallosal",
        "G_and_S_cingul-Ant", "G_insular_short", "G_Ins_lg_and_S_cent_ins",
        "S_circular_insula_ant", "S_circular_insula_inf", "S_circular_insula_sup",
    ],
    "Default mode (self & memory)": [
        "G_precuneus", "G_cingul-Post-dorsal", "G_cingul-Post-ventral",
        "S_subparietal", "G_oc-temp_med-Parahip",
    ],
}
NETWORK_COLORS = {
    "Overall": "#111111",
    "Visual": "#1f77b4",
    "Faces & objects": "#17becf",
    "Auditory": "#ff7f0e",
    "Language": "#d62728",
    "Attention": "#9467bd",
    "Motor & touch": "#8c564b",
    "Prefrontal": "#7f7f7f",
    "Emotion & reward": "#e377c2",
    "Default mode (self & memory)": "#2ca02c",
}


# ----------------------------------------------------------------------
# Mesh and atlas
# ----------------------------------------------------------------------


@functools.lru_cache(maxsize=1)
def load_mesh() -> dict:
    """Half-inflated fsaverage5 mesh (as in the TRIBE plotting code) + sulcal depth."""
    from nilearn import datasets, surface

    fs = datasets.fetch_surf_fsaverage("fsaverage5")
    out = {}
    for hemi in ("left", "right"):
        pial = surface.load_surf_mesh(fs[f"pial_{hemi}"])
        infl = surface.load_surf_mesh(fs[f"infl_{hemi}"])
        coords = 0.5 * np.asarray(pial.coordinates) + 0.5 * np.asarray(infl.coordinates)
        # pull the hemispheres apart so they never overlap in the 3D view
        if hemi == "left":
            coords[:, 0] -= coords[:, 0].max() + 5
        else:
            coords[:, 0] -= coords[:, 0].min() - 5
        sulc = np.asarray(surface.load_surf_data(fs[f"sulc_{hemi}"]), dtype=float)
        out[hemi] = dict(coords=coords, faces=np.asarray(infl.faces), sulc=sulc)
    return out


@functools.lru_cache(maxsize=1)
def load_network_masks() -> dict[str, np.ndarray] | None:
    """Boolean vertex masks (length 2*N_VERTS_HEMI) for each network in NETWORKS.

    Returns None if the Destrieux atlas cannot be downloaded.
    """
    from nilearn import datasets

    try:
        atlas = datasets.fetch_atlas_surf_destrieux()
    except Exception as e:  # network-restricted environments
        logger.warning("Could not fetch Destrieux atlas (%s); networks disabled", e)
        return None
    names = [n.decode() if isinstance(n, bytes) else str(n) for n in atlas["labels"]]
    labels = np.r_[np.asarray(atlas["map_left"]), np.asarray(atlas["map_right"])]
    name_to_idx = {n: i for i, n in enumerate(names)}
    masks = {}
    for net, parcels in NETWORKS.items():
        idx = [name_to_idx[p] for p in parcels if p in name_to_idx]
        masks[net] = np.isin(labels, idx)
    cortex = ~np.isin(labels, [name_to_idx.get("Medial_wall", -1), name_to_idx.get("Unknown", 0)])
    masks["_cortex"] = cortex
    return masks


# ----------------------------------------------------------------------
# Summaries
# ----------------------------------------------------------------------


# Same lines and axis on every reel's charts, so different reels can be compared.
COMPARE_NETWORKS = ["Visual", "Auditory", "Language", "Faces & objects", "Attention"]
CHART_YLIM = (-0.15, 0.35)


def chart_ylim(ts: dict) -> tuple[float, float]:
    """Fixed y-range, widened only if a reel goes beyond it."""
    vals = np.concatenate([s[~np.isnan(s)] for s in ts.values()] or [np.zeros(1)])
    lo, hi = CHART_YLIM
    if vals.size:
        lo, hi = min(lo, float(vals.min()) - 0.02), max(hi, float(vals.max()) + 0.02)
    return lo, hi


def densify(preds: np.ndarray, starts: list[float], tr: float, n_steps: int | None = None) -> np.ndarray:
    """Place per-segment predictions on a regular 0, TR, 2TR... grid (NaN where missing)."""
    idx = np.round(np.asarray(starts) / tr).astype(int)
    n = max(int(idx.max()) + 1, n_steps or 0)
    dense = np.full((n, preds.shape[1]), np.nan, dtype=np.float32)
    dense[idx] = preds
    return dense


def network_timeseries(dense: np.ndarray, masks: dict | None) -> dict[str, np.ndarray]:
    cortex = masks["_cortex"] if masks else np.ones(dense.shape[1], bool)
    out = {"Overall": np.nanmean(dense[:, cortex], axis=1) if len(dense) else np.array([])}
    if masks:
        for net in NETWORKS:
            out[net] = np.nanmean(dense[:, masks[net]], axis=1)
    return out


def find_peaks(series: np.ndarray, k: int = 3, min_gap: int = 3) -> list[int]:
    order = np.argsort(np.nan_to_num(series, nan=-np.inf))[::-1]
    chosen: list[int] = []
    for i in order:
        if np.isnan(series[i]):
            continue
        if all(abs(i - j) >= min_gap for j in chosen):
            chosen.append(int(i))
        if len(chosen) == k:
            break
    return chosen


def summary_table(ts: dict[str, np.ndarray], tr: float) -> list[list]:
    nets = [n for n in ts if n != "Overall"]
    winners = None
    if nets:
        stack = np.vstack([ts[n] for n in nets])
        valid = ~np.isnan(stack).any(0)
        winners = np.full(stack.shape[1], -1)
        winners[valid] = np.argmax(stack[:, valid], axis=0)
    rows = []
    for j, (name, s) in enumerate(ts.items()):
        if np.all(np.isnan(s)):
            continue
        peak_i = int(np.nanargmax(s))
        lead = "" if name == "Overall" or winners is None else (
            f"{100 * np.mean(winners[winners >= 0] == nets.index(name)):.0f}%"
        )
        rows.append([name, round(float(np.nanmean(s)), 4), round(float(np.nanmax(s)), 4),
                     round(peak_i * tr, 1), lead])
    return rows


def vertex_limits(dense: np.ndarray, percentile: float = 99.0, threshold_pct: float = 60.0):
    """Shared color limits across all timesteps so frames are comparable."""
    vals = np.abs(dense[~np.isnan(dense)])
    vmax = float(np.percentile(vals, percentile)) if vals.size else 1.0
    thr = vmax * threshold_pct / 100.0
    return thr, max(vmax, 1e-6)


# ----------------------------------------------------------------------
# Fast 2D rendering (matplotlib polygons with back-face culling)
# ----------------------------------------------------------------------


class BrainRenderer:
    """Renders four standard views quickly by projecting the mesh once and
    only updating face colours per frame."""

    def __init__(self, views=tuple(VIEWS)):
        self.mesh = load_mesh()
        self.views = list(views)
        self._geom = {v: self._project(*VIEWS[v]) for v in self.views}

    def _project(self, hemi: str, side: int):
        m = self.mesh[hemi]
        tri = m["coords"][m["faces"]]  # (F, 3, 3)
        normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
        normals /= np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12
        facing = normals[:, 0] * side
        keep = facing > 0
        depth = (tri[:, :, 0].mean(1) * side)[keep]
        order = np.argsort(depth)  # far first (painter's algorithm)
        faces_idx = np.flatnonzero(keep)[order]
        xy = np.stack([-side * tri[faces_idx, :, 1], tri[faces_idx, :, 2]], axis=-1)
        shade = 0.45 + 0.55 * facing[faces_idx]
        sulc = m["sulc"][m["faces"][faces_idx]].mean(1)
        base = np.where(sulc > 0, 0.55, 0.78)
        offset = 0 if hemi == "left" else N_VERTS_HEMI
        return dict(xy=xy, faces=m["faces"][faces_idx] + offset, shade=shade, base=base)

    def face_colors(self, view: str, values: np.ndarray, thr: float, vmax: float, diverging: bool):
        g = self._geom[view]
        rgb = np.repeat(g["base"][:, None], 3, axis=1)
        if values is not None and not np.all(np.isnan(values)):
            fv = np.nan_to_num(values[g["faces"]].mean(1))
            rgb = overlay(rgb, fv, thr, vmax, diverging)
        return np.clip(rgb * g["shade"][:, None], 0, 1)

    def draw(self, ax, view: str, values, thr, vmax, diverging=False, title=True):
        g = self._geom[view]
        pc = PolyCollection(g["xy"], facecolors=self.face_colors(view, values, thr, vmax, diverging),
                            edgecolors="face", linewidths=0.15, antialiaseds=False)
        ax.add_collection(pc)
        ax.set_xlim(g["xy"][..., 0].min() - 2, g["xy"][..., 0].max() + 2)
        ax.set_ylim(g["xy"][..., 1].min() - 2, g["xy"][..., 1].max() + 2)
        ax.set_aspect("equal")
        ax.axis("off")
        if title:
            ax.set_title(view, fontsize=9, color="#444")
        return pc


def _cmap(diverging: bool):
    return plt.get_cmap("RdBu_r" if diverging else "hot")


def overlay(rgb: np.ndarray, v: np.ndarray, thr: float, vmax: float, diverging: bool) -> np.ndarray:
    rgb = rgb.copy()
    if diverging:
        mask = np.abs(v) > thr
        x = np.clip(0.5 + 0.5 * v[mask] / vmax, 0, 1)
    else:
        mask = v > thr
        # map thr..vmax to the bright part of "hot" (dark red -> yellow/white)
        x = 0.25 + 0.75 * np.clip((v[mask] - thr) / max(vmax - thr, 1e-9), 0, 1)
    rgb[mask] = _cmap(diverging)(x)[:, :3]
    return rgb


def colorbar_mappable(thr, vmax, diverging):
    if diverging:
        norm = matplotlib.colors.Normalize(-vmax, vmax)
    else:
        norm = matplotlib.colors.Normalize(thr - (vmax - thr) / 3, vmax)
    return matplotlib.cm.ScalarMappable(norm=norm, cmap=_cmap(diverging))


def render_static(renderer: BrainRenderer, values, thr, vmax, diverging, title: str, path: str):
    fig, axes = plt.subplots(2, 2, figsize=(8, 6.2), dpi=110)
    for ax, view in zip(axes.flat, renderer.views):
        renderer.draw(ax, view, values, thr, vmax, diverging)
    fig.suptitle(title, fontsize=13, fontweight="bold")
    cax = fig.add_axes([0.25, 0.04, 0.5, 0.02])
    fig.colorbar(colorbar_mappable(thr, vmax, diverging), cax=cax, orientation="horizontal",
                 label="predicted response (a.u.)")
    fig.subplots_adjust(left=0.02, right=0.98, top=0.9, bottom=0.1, wspace=0.02, hspace=0.12)
    fig.savefig(path)
    plt.close(fig)
    return path


class PanelAnimator:
    """Per-timestep panel: 4 brain views + intensity timeline with a cursor."""

    def __init__(self, renderer: BrainRenderer, ts: dict, tr: float, thr, vmax, diverging,
                 width_px=640, height_px=720, unreliable_last=False):
        self.r, self.thr, self.vmax, self.div = renderer, thr, vmax, diverging
        dpi = 100
        self.fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi, facecolor="white")
        gs = self.fig.add_gridspec(2, 2, hspace=0.12, wspace=0.02,
                                   left=0.03, right=0.97, top=0.9, bottom=0.33)
        self.pcs = {}
        for k, view in enumerate(renderer.views):
            ax = self.fig.add_subplot(gs[k // 2, k % 2])
            self.pcs[view] = renderer.draw(ax, view, None, thr, vmax, diverging)
        ax = self.fig.add_axes([0.1, 0.07, 0.86, 0.22])
        t = np.arange(len(ts["Overall"])) * tr
        for name in ["Overall"] + [n for n in COMPARE_NETWORKS if n in ts]:
            ax.plot(t, ts[name], color=NETWORK_COLORS.get(name, None),
                    lw=2.2 if name == "Overall" else 1.2, label=name)
        ax.set_ylim(*chart_ylim(ts))
        if unreliable_last and len(t) > 1:
            ax.axvspan(t[-1] - tr / 2, t[-1] + tr / 2, color="#999", alpha=0.25, lw=0)
        ax.legend(fontsize=7, ncol=3, loc="upper left", frameon=False)
        ax.set_xlabel("time (s)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        self.cursor = ax.axvline(0, color="#e4572e", lw=2)
        self.title = self.fig.text(0.5, 0.955, "", ha="center", va="center", fontsize=13,
                                   fontweight="bold")
        self.ts, self.tr = ts, tr

    def frame(self, values: np.ndarray, i: int) -> np.ndarray:
        for view, pc in self.pcs.items():
            pc.set_facecolor(self.r.face_colors(view, values, self.thr, self.vmax, self.div))
        self.cursor.set_xdata([i * self.tr, i * self.tr])
        nets = {n: s[i] for n, s in self.ts.items() if n != "Overall" and not np.isnan(s[i])}
        top = ", ".join(sorted(nets, key=nets.get, reverse=True)[:2])
        label = f"t = {i * self.tr:.0f}s" + (f"  ·  top: {top}" if top else "")
        if np.all(np.isnan(values)):
            label = f"t = {i * self.tr:.0f}s  ·  (no prediction)"
        self.title.set_text(label)
        self.fig.canvas.draw()
        return np.asarray(self.fig.canvas.buffer_rgba())[..., :3].copy()

    def close(self):
        plt.close(self.fig)


# ----------------------------------------------------------------------
# Plotly figures
# ----------------------------------------------------------------------


def _plotly_scale(values: np.ndarray, base: np.ndarray, thr: float, vmax: float, diverging: bool):
    """Encode background (sulcal gray) and activation into one 0..1 intensity
    so a single colorscale can show both: [0, .2) = anatomy, [.2, 1] = activation."""
    out = 0.2 * base
    v = np.nan_to_num(values)
    if diverging:
        m = np.abs(v) > thr
        out[m] = 0.2 + 0.8 * np.clip(0.5 + 0.5 * v[m] / vmax, 0, 0.999)
    else:
        m = v > thr
        out[m] = 0.2 + 0.8 * np.clip((v[m] - thr) / max(vmax - thr, 1e-9), 0, 1)
    return out.astype(np.float32)


def _plotly_colorscale(diverging: bool):
    cm = _cmap(diverging)
    scale = [[0.0, "rgb(120,120,120)"], [0.1999, "rgb(205,205,205)"]]
    for x in np.linspace(0, 1, 9):
        r, g, b, _ = cm(x if diverging else 0.25 + 0.75 * x)
        scale.append([0.2 + 0.8 * x, f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"])
    return scale


CAMERAS = {
    "Left": dict(x=-1.7, y=0, z=0.2),
    "Right": dict(x=1.7, y=0, z=0.2),
    "Top": dict(x=0, y=-0.01, z=1.9),
    "Front": dict(x=0, y=1.9, z=0.2),
    "Back": dict(x=0, y=-1.9, z=0.2),
}


def brain_3d_figure(dense: np.ndarray, tr: float, thr: float, vmax: float, diverging: bool,
                    mean_map: np.ndarray | None = None):
    import plotly.graph_objects as go

    mesh = load_mesh()
    coords = np.r_[mesh["left"]["coords"], mesh["right"]["coords"]]
    faces = np.r_[mesh["left"]["faces"], mesh["right"]["faces"] + N_VERTS_HEMI]
    sulc = np.r_[mesh["left"]["sulc"], mesh["right"]["sulc"]]
    base = np.where(sulc > 0, 0.25, 0.95)

    frames_vals = []
    if mean_map is not None:
        frames_vals.append(("average", "Average", mean_map))
    for i in range(len(dense)):
        frames_vals.append((str(i), f"{i * tr:.0f}s", dense[i]))

    first = _plotly_scale(frames_vals[0][2], base, thr, vmax, diverging)
    trace = go.Mesh3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        intensity=first, cmin=0, cmax=1, colorscale=_plotly_colorscale(diverging),
        showscale=False, hoverinfo="skip", flatshading=False,
        lighting=dict(ambient=0.55, diffuse=0.7, specular=0.05, roughness=0.9),
        lightposition=dict(x=0, y=0, z=1e5),
    )
    frames = [go.Frame(name=name, data=[go.Mesh3d(intensity=_plotly_scale(v, base, thr, vmax, diverging))],
                       traces=[0]) for name, _, v in frames_vals]
    steps = [dict(method="animate", label=label,
                  args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                     transition=dict(duration=0))])
             for name, label, _ in frames_vals]
    play_frames = [name for name, _, _ in frames_vals if name != "average"]
    fig = go.Figure(data=[trace], frames=frames)
    fig.update_layout(
        height=660, margin=dict(l=0, r=0, t=10, b=0), paper_bgcolor="white",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
                   aspectmode="data", camera=dict(eye=dict(x=-1.6, y=0, z=0.3))),
        sliders=[dict(active=0, steps=steps, x=0.05, len=0.9, y=0.02,
                      currentvalue=dict(prefix="Showing: "))],
        updatemenus=[dict(type="buttons", direction="left", x=0.05, y=1.0, showactive=False,
                          buttons=[dict(label=lbl, method="relayout", args=[{"scene.camera.eye": eye}])
                                   for lbl, eye in CAMERAS.items()]),
                     dict(type="buttons", direction="left", x=0.05, y=0.12, showactive=False,
                          buttons=[
                              dict(label="▶ Play", method="animate",
                                   args=[play_frames, dict(frame=dict(duration=int(tr * 1000), redraw=True),
                                                           transition=dict(duration=0), fromcurrent=True,
                                                           mode="immediate")]),
                              dict(label="❚❚ Pause", method="animate",
                                   args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))]),
                          ])],
    )
    return fig


def intensity_figure(ts: dict[str, np.ndarray], tr: float, peaks: list[int],
                     unreliable_last: bool = False):
    import plotly.graph_objects as go

    fig = go.Figure()
    t = np.arange(len(ts["Overall"])) * tr
    for name, s in ts.items():
        fig.add_trace(go.Scatter(
            x=t, y=s, name=name, mode="lines",
            line=dict(color=NETWORK_COLORS.get(name), width=4 if name == "Overall" else 2),
            visible=True if name in ["Overall"] + COMPARE_NETWORKS else "legendonly",
        ))
    if unreliable_last and len(t) > 1:
        fig.add_vrect(x0=t[-1] - tr / 2, x1=t[-1] + tr / 2, fillcolor="#999", opacity=0.25,
                      line_width=0, annotation_text="end of video: less reliable",
                      annotation_position="bottom right")
    for rank, i in enumerate(peaks, 1):
        fig.add_vline(x=i * tr, line=dict(color="#e4572e", dash="dot"),
                      annotation_text=f"peak #{rank}", annotation_position="top")
    fig.update_layout(
        height=480, margin=dict(l=50, r=20, t=40, b=40), hovermode="x unified",
        xaxis_title="time in reel (s)", yaxis_title="predicted response (a.u.)",
        legend=dict(orientation="h", y=-0.2), template="plotly_white",
        yaxis=dict(range=list(chart_ylim(ts))),
        title=dict(text="Brain response intensity over time (click legend items to toggle)",
                   font=dict(size=14)),
    )
    return fig

In [ ]:
%%writefile app.py
"""TRIBE v2 Reel Studio: upload a reel, see the predicted brain response.

Run:
    python app.py              # real TRIBE v2 model (GPU strongly recommended)
    python app.py --share      # also create a public gradio.live link (Colab)
    python app.py --demo       # synthetic predictions, for trying the UI without a GPU
"""

from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
sys.path.insert(0, str(Path(__file__).resolve().parent))

import gradio as gr  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

import brain_viz as bv  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("reel_studio")

CACHE = Path(os.environ.get("TRIBE_CACHE", "./cache")).resolve()
OUT_ROOT = CACHE / "reel_outputs"
# Fixed colour scale shared by every reel (predicted response, a.u.), so brain maps of
# different reels mean the same thing. Adjustable in the UI.
DEFAULT_SCALE_MAX = 0.5
# Extra seconds analysed past the requested clip and then discarded: the model's last
# second is unreliable because it has no footage after it.
EDGE_PAD_S = 2.0
LANGUAGES = ["english", "french", "spanish", "dutch", "chinese"]
DEMO = False
_MODEL = None


# ----------------------------------------------------------------------
# Video helpers
# ----------------------------------------------------------------------


def probe_duration(path: str) -> float:
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", path],
        capture_output=True, text=True, check=True,
    )
    return float(json.loads(out.stdout)["format"]["duration"])


def has_audio_stream(path: str) -> bool:
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index",
         "-of", "csv=p=0", path], capture_output=True, text=True,
    )
    return bool(out.stdout.strip())


IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def image_to_video(path: str, seconds: float) -> str:
    """Turn a still photo into a silent clip (TRIBE only takes video/audio/text)."""
    out = CACHE / "uploads" / f"{Path(path).stem}_{int(seconds)}s.mp4"
    out.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", path, "-t", str(seconds),
                    "-vf", "scale='min(1080,iw)':-2,format=yuv420p", "-r", "25", "-c:v", "libx264",
                    str(out)], check=True)
    return str(out)


def stage_upload(path: str, start: float = 0, seconds: float = 0) -> Path:
    """Copy the upload to a stable, content-addressed .mp4 so features get cached
    and re-analysing the same reel is fast. Non-mp4 inputs are transcoded.
    With start/seconds set, only that part of the video is kept, so the model
    only processes those seconds."""
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    trim = start > 0 or seconds > 0
    suffix = f"_from{start:g}s_{seconds:g}s" if trim else ""
    dst = CACHE / "uploads" / f"reel_{h.hexdigest()[:16]}{suffix}.mp4"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        if trim:
            cut = ["-ss", f"{start:g}"] + (["-t", f"{seconds:g}"] if seconds > 0 else [])
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *cut, "-i", path, "-c:v", "libx264",
                            "-pix_fmt", "yuv420p", "-c:a", "aac", str(dst)], check=True)
        elif Path(path).suffix.lower() == ".mp4":
            shutil.copy(path, dst)
        else:
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-c:v", "libx264",
                            "-pix_fmt", "yuv420p", "-c:a", "aac", str(dst)], check=True)
    return dst


def extract_frames(video: Path, times: list[float], out_dir: Path, height=360) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for t in times:
        p = out_dir / f"frame_{t:07.2f}.jpg"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{t:.2f}", "-i", str(video),
                        "-frames:v", "1", "-vf", f"scale=-2:{height}", str(p)], check=False)
        paths.append(p)
    return paths


# ----------------------------------------------------------------------
# TRIBE inference
# ----------------------------------------------------------------------


WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "large-v3-turbo")
WHISPER_LANG = dict(english="en", french="fr", spanish="es", dutch="nl", chinese="zh")
_WHISPER = None


def transcribe_words(wav_filename, language: str) -> pd.DataFrame:
    """Drop-in for TRIBE's WhisperX step: faster-whisper in-process, with the same output
    columns. TRIBE runs WhisperX large-v3 through `uvx`, which rebuilds a separate
    environment on every container start; this skips that and uses the faster turbo model."""
    global _WHISPER
    import torch
    from faster_whisper import WhisperModel

    if language not in WHISPER_LANG:
        raise ValueError(f"Language {language} not supported")
    if _WHISPER is None:
        cuda = torch.cuda.is_available()
        _WHISPER = WhisperModel(WHISPER_MODEL, device="cuda" if cuda else "cpu",
                                compute_type="float16" if cuda else "int8")
    segments, _ = _WHISPER.transcribe(str(wav_filename), language=WHISPER_LANG[language],
                                      word_timestamps=True, vad_filter=True)
    rows = []
    for i, seg in enumerate(segments):
        sentence = seg.text.strip().replace('"', "")
        for w in seg.words or []:
            rows.append(dict(text=w.word.strip().replace('"', ""), start=w.start,
                             duration=w.end - w.start, sequence_id=i, sentence=sentence))
    return pd.DataFrame(rows, columns=["text", "start", "duration", "sequence_id", "sentence"])


def warm_up():
    """Fetch LLaMA 3.2 (gated, so it can't be preloaded at build time) and load the models
    in the background while the UI is already up, so the first upload doesn't wait for it."""
    try:
        from huggingface_hub import snapshot_download

        snapshot_download("meta-llama/Llama-3.2-3B", allow_patterns=["*.json", "*.safetensors"])
        get_model()
        logger.info("Warm-up done: models ready.")
    except Exception:
        logger.exception("Background warm-up failed; models will load on first analysis")


def get_model():
    global _MODEL
    if _MODEL is None:
        import torch
        from tribev2.demo_utils import TribeModel

        from tribev2.eventstransforms import ExtractWordsFromAudio

        if torch.cuda.is_available():
            # TF32 tensor cores for float32 matmuls/convs: large speedup on Ampere+ GPUs
            # (L40S, A100), negligible accuracy cost.
            torch.set_float32_matmul_precision("high")
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        else:
            logger.warning("No CUDA GPU detected: inference will be very slow on CPU.")
        ExtractWordsFromAudio._get_transcript_from_audio = staticmethod(transcribe_words)
        _MODEL = TribeModel.from_pretrained("facebook/tribev2", cache_folder=str(CACHE))
    return _MODEL


def build_events(video: Path, language: str) -> tuple[pd.DataFrame, str]:
    """Same pipeline as TribeModel.get_events_dataframe, with a language choice and
    a fallback for reels without intelligible speech (music-only, etc.)."""
    from neuralset.events.transforms import (AddContextToWords, AddSentenceToWords, AddText,
                                             ChunkEvents, ExtractAudioFromVideo, RemoveMissing)
    from neuralset.events.utils import standardize_events
    from tribev2.eventstransforms import ExtractWordsFromAudio

    base = [
        ExtractAudioFromVideo(),
        ChunkEvents(event_type_to_chunk="Audio", max_duration=60, min_duration=30),
        ChunkEvents(event_type_to_chunk="Video", max_duration=60, min_duration=30),
    ]
    text = [
        ExtractWordsFromAudio(language=language),
        AddText(),
        AddSentenceToWords(max_unmatched_ratio=0.05),
        AddContextToWords(sentence_only=False, max_context_len=1024, split_field=""),
        RemoveMissing(),
    ]
    event = {"type": "Video", "filepath": str(video), "start": 0,
             "timeline": "default", "subject": "default"}

    def run(transforms):
        ev = standardize_events(pd.DataFrame([event]))
        for t in transforms:
            ev = t(ev)
        return standardize_events(ev)

    try:
        events = run(base + text)
        n_words = int((events.type == "Word").sum())
        if n_words:
            return events, f"Transcribed {n_words} words ({language})."
        note = "No speech detected: using video + audio only."
    except Exception as e:  # e.g. music-only reel, transcription failure
        logger.exception("Transcription stage failed")
        note = f"Speech transcription failed ({type(e).__name__}); using video + audio only."
    return run(base), note


def predict_tribe(video: Path, language: str, progress) -> tuple[np.ndarray, list[float], float, str]:
    progress(0.05, desc="Loading TRIBE v2 (first run downloads weights)…")
    model = get_model()
    progress(0.15, desc="Extracting audio + transcribing speech…")
    events, note = build_events(video, language)
    progress(0.3, desc="Extracting V-JEPA2 / Wav2Vec-BERT / LLaMA features + predicting…")
    preds, segments = model.predict(events=events, verbose=True)
    starts = [float(s.start) for s in segments]
    return preds.astype(np.float32), starts, float(model.data.TR), note


def predict_demo(video: Path, duration: float) -> tuple[np.ndarray, list[float], float, str]:
    """Synthetic but structured predictions for UI testing. NOT brain data."""
    rng = np.random.default_rng(0)
    tr = 1.0
    n = max(int(duration // tr), 1)
    masks = bv.load_network_masks()
    if masks is None:  # offline fallback: coarse spatial blobs from mesh geometry
        mesh = bv.load_mesh()
        coords = np.r_[mesh["left"]["coords"], mesh["right"]["coords"]]
        centers = coords[rng.choice(len(coords), len(bv.NETWORKS), replace=False)]
        lab = np.argmin(((coords[:, None] - centers[None]) ** 2).sum(-1), axis=1)
        masks = {net: lab == k for k, net in enumerate(bv.NETWORKS)}
    preds = 0.05 * rng.standard_normal((n, 2 * bv.N_VERTS_HEMI)).astype(np.float32)
    t = np.arange(n)
    for k, net in enumerate(bv.NETWORKS):
        tc = np.convolve(rng.standard_normal(n + 6), np.hanning(7), "same")[3:n + 3]
        tc = 0.25 * tc + 0.15 * np.sin(2 * np.pi * t / (6 + k))
        preds[:, masks[net]] += tc[:, None].astype(np.float32)
    return preds, list(t * tr), tr, "DEMO MODE: synthetic data, not TRIBE predictions."


# ----------------------------------------------------------------------
# Outputs
# ----------------------------------------------------------------------


def make_brain_movie(video: Path, dense, ts, tr, thr, vmax, diverging, out_dir: Path, progress,
                     unreliable_last=False) -> Path:
    renderer = bv.BrainRenderer()
    anim = bv.PanelAnimator(renderer, ts, tr, thr, vmax, diverging, unreliable_last=unreliable_last)
    frames_dir = out_dir / "panels"
    frames_dir.mkdir(parents=True, exist_ok=True)
    from PIL import Image

    for i in range(len(dense)):
        progress(0.6 + 0.3 * i / max(len(dense), 1), desc=f"Rendering brain movie {i + 1}/{len(dense)}")
        Image.fromarray(anim.frame(dense[i], i)).save(frames_dir / f"p_{i:05d}.png")
    anim.close()

    out = out_dir / "reel_with_brain.mp4"
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video),
           "-framerate", f"{1 / tr}", "-start_number", "0", "-i", str(frames_dir / "p_%05d.png"),
           "-filter_complex",
           "[0:v]scale=-2:720,setsar=1,fps=25[a];[1:v]scale=-2:720,setsar=1,fps=25[b];"
           "[a][b]hstack=inputs=2,format=yuv420p[v]",
           "-map", "[v]"]
    if has_audio_stream(str(video)):
        cmd += ["-map", "0:a", "-c:a", "aac"]
    # Cut to the analysed seconds (the staged video may carry discarded padding seconds).
    cmd += ["-t", f"{len(dense) * tr:g}", "-c:v", "libx264", "-crf", "20", "-preset", "veryfast",
            "-movflags", "+faststart", str(out)]
    subprocess.run(cmd, check=True)
    return out


def make_moments(video: Path, dense, ts, peaks, tr, thr, vmax, diverging, out_dir: Path):
    import matplotlib.pyplot as plt
    from PIL import Image

    renderer = bv.BrainRenderer(views=("Left lateral", "Right lateral"))
    frames = extract_frames(video, [i * tr + tr / 2 for i in peaks], out_dir / "frames")
    gallery = []
    for rank, (i, fp) in enumerate(zip(peaks, frames), 1):
        fig = plt.figure(figsize=(9, 3.6), dpi=100)
        gs = fig.add_gridspec(1, 3, width_ratios=[0.8, 1, 1], wspace=0.02)
        ax = fig.add_subplot(gs[0])
        if fp.exists():
            ax.imshow(Image.open(fp))
        ax.axis("off")
        for k, view in enumerate(renderer.views):
            renderer.draw(fig.add_subplot(gs[k + 1]), view, dense[i], thr, vmax, diverging)
        nets = {n: s[i] for n, s in ts.items() if n != "Overall"}
        top = ", ".join(sorted(nets, key=nets.get, reverse=True)[:3])
        caption = f"#{rank} · t = {i * tr:.0f}s" + (f" · strongest: {top}" if top else "")
        fig.suptitle(caption, fontsize=11, fontweight="bold")
        p = out_dir / f"moment_{rank}.png"
        fig.savefig(p, bbox_inches="tight")
        plt.close(fig)
        gallery.append((str(p), caption))
    return gallery


def analyze(video_path, image_path, image_seconds, clip_start, clip_seconds, language, colormap,
            threshold_pct, scale_max, make_movie, progress=gr.Progress()):
    if not video_path and image_path:
        # Render the photo a bit longer than asked and analyse only the asked seconds.
        video_path = image_to_video(image_path, image_seconds + EDGE_PAD_S)
        clip_start, clip_seconds = 0, image_seconds
    if not video_path:
        raise gr.Error("Upload a reel or a photo first.")
    t0 = time.time()
    progress(0.01, desc="Preparing video…")
    clip_start, clip_seconds = float(clip_start or 0), float(clip_seconds or 0)
    if clip_start > 0 and clip_start >= probe_duration(video_path):
        raise gr.Error(f"'Start at' ({clip_start:g}s) is past the end of the video.")
    video = stage_upload(video_path, clip_start, clip_seconds + EDGE_PAD_S if clip_seconds else 0)
    duration = probe_duration(str(video))
    keep_s = min(clip_seconds, duration) if clip_seconds else duration
    if duration > 300:
        raise gr.Error(f"Video is {duration:.0f}s long; please keep it under 5 minutes.")

    if DEMO:
        preds, starts, tr, note = predict_demo(video, duration)
    else:
        preds, starts, tr, note = predict_tribe(video, language, progress)

    progress(0.55, desc="Summarizing…")
    diverging = colormap.startswith("Activation & suppression")
    n_keep = int(np.ceil(keep_s / tr - 1e-6))
    dense = bv.densify(preds, starts, tr, n_steps=int(np.ceil(duration / tr)))
    # The padding seconds only exist to give the kept seconds footage after them.
    unreliable_last = len(dense) <= n_keep
    dense = dense[:n_keep]
    masks = bv.load_network_masks()
    ts = bv.network_timeseries(dense, masks)
    # Peaks and the summary table skip an unreliable final second.
    ts_eval = {k: v[:-1] for k, v in ts.items()} if unreliable_last and n_keep > 1 else ts
    peaks = bv.find_peaks(ts_eval["Overall"], k=3)
    _, own_p99 = bv.vertex_limits(dense)
    vmax = float(scale_max or DEFAULT_SCALE_MAX)
    thr = vmax * threshold_pct / (200 if diverging else 100)
    mean_map = np.nanmean(dense, axis=0)

    out_dir = OUT_ROOT / f"{video.stem}_{int(time.time())}"
    out_dir.mkdir(parents=True, exist_ok=True)

    avg_png = bv.render_static(bv.BrainRenderer(), mean_map, thr, vmax, diverging,
                               "Average predicted activation over the whole reel",
                               str(out_dir / "average_activation.png"))
    gallery = make_moments(video, dense, ts, peaks, tr, thr, vmax, diverging, out_dir)
    fig3d = bv.brain_3d_figure(dense, tr, thr, vmax, diverging, mean_map=mean_map)
    html_3d = out_dir / "brain_3d_interactive.html"
    fig3d.write_html(str(html_3d), include_plotlyjs=True, auto_play=False,
                     default_height="100%", default_width="100%")
    viewer = (f'<iframe src="/gradio_api/file={html_3d}" style="width:100%;height:680px;border:0" '
              f'title="Interactive 3D brain"></iframe>')
    fig_int = bv.intensity_figure(ts, tr, peaks, unreliable_last)

    movie = None
    if make_movie:
        movie = make_brain_movie(video, dense, ts, tr, thr, vmax, diverging, out_dir, progress,
                                 unreliable_last)

    table = pd.DataFrame(bv.summary_table(ts_eval, tr),
                         columns=["Region / network", "Mean", "Peak", "Peak at (s)", "Top network % of time"])
    np.save(out_dir / "predictions_fsaverage5.npy", dense)
    ts_df = pd.DataFrame({"time_s": np.arange(len(dense)) * tr, **ts})
    ts_df.to_csv(out_dir / "intensity_timeseries.csv", index=False)
    table.to_csv(out_dir / "network_summary.csv", index=False)
    files = [str(out_dir / f) for f in ("intensity_timeseries.csv", "network_summary.csv",
                                        "predictions_fsaverage5.npy", "brain_3d_interactive.html")]
    if movie:
        files.insert(0, str(movie))

    status = (f"**Done in {time.time() - t0:.0f}s.** {note}  \n"
              f"{len(dense)} timesteps over a {keep_s:.1f}s clip (1 step = {tr:g}s; "
              f"5s hemodynamic lag already compensated).  \n"
              f"Colour scale max: {vmax:g} (this reel's own 99th percentile: {own_p99:.2f}). "
              f"Keep the scale the same when comparing reels.")
    if unreliable_last:
        status += ("  \n⚠️ No footage after the last second, so it's less reliable: it is shaded "
                   "and left out of peaks and the summary.")
    elif clip_seconds:
        status += f"  \n{EDGE_PAD_S:g} extra seconds were analysed past the clip and discarded."
    if masks is None:
        status += "  \n⚠️ Destrieux atlas unavailable (no internet?), so network breakdown is disabled."
    if DEMO:
        status = "### ⚠️ DEMO MODE: synthetic data, NOT real TRIBE v2 predictions\n" + status
    return (status, str(movie) if movie else None, viewer, fig_int, avg_png, gallery, table, files)


# ----------------------------------------------------------------------
# UI
# ----------------------------------------------------------------------

INTRO = """
# 🧠 TRIBE v2 Reel Studio
Upload a reel (vertical or horizontal video). **TRIBE v2** (Meta FAIR) predicts the fMRI response of an
*average human brain* watching it, second by second, on ~20k cortical vertices (fsaverage5).

<small>Predictions are model estimates for an average viewer, not measurements of real people. Network groupings are
anatomical approximations (Destrieux atlas). TRIBE v2 is licensed CC BY-NC 4.0 (non-commercial use only).</small>
"""

HELP = """
**How to read the results**
- **Brain movie**: your reel next to the predicted brain response. Brighter colors mean a stronger predicted response.
- **3D brain**: rotate the brain, scrub through time with the slider, or pick *Average*.
- **Intensity**: the response over time, for the whole cortex and for each brain system. The dotted lines mark the 3 strongest moments.
- **Key moments**: the frames where the overall response peaked, with the brain map at each one.
- **Visual** = what's on screen · **Auditory** = sound/music · **Language** = speech and meaning ·
  **Faces & objects** = people and things · **Emotion & reward** = OFC/ACC/insula (rough proxy).
"""


def build_ui() -> gr.Blocks:
    with gr.Blocks(title="TRIBE v2 Reel Studio") as demo:
        gr.Markdown(INTRO)
        if DEMO:
            gr.Markdown("### ⚠️ Running in DEMO mode: outputs are synthetic, for UI testing only.")
        with gr.Row():
            with gr.Column(scale=1, min_width=320):
                video_in = gr.Video(label="Your reel (.mp4 / .mov / .webm)", sources=["upload"], height=480)
                with gr.Row():
                    clip_start = gr.Number(value=0, minimum=0, label="Start at (s)")
                    clip_seconds = gr.Number(value=0, minimum=0,
                                             label="Seconds to analyze (0 = whole video)")
                with gr.Accordion("…or analyze a photo", open=False):
                    image_in = gr.Image(label="Photo (shown as a still clip)", type="filepath", sources=["upload"])
                    image_seconds = gr.Slider(5, 30, value=10, step=1, label="Show the photo for (s)")
                language = gr.Dropdown(LANGUAGES, value="english", label="Spoken language in the reel")
                with gr.Accordion("Display options", open=False):
                    colormap = gr.Radio(["Activation only (hot)", "Activation & suppression (blue/red)"],
                                        value="Activation only (hot)", label="Color map")
                    threshold = gr.Slider(0, 95, value=55, step=5,
                                          label="Hide weak responses below (% of scale max)")
                    scale_max = gr.Number(value=DEFAULT_SCALE_MAX, minimum=0.01,
                                          label="Color scale max (keep the same to compare reels)")
                    make_movie = gr.Checkbox(value=True, label="Render side-by-side brain movie")
                run_btn = gr.Button("🧠 Analyze reel", variant="primary", size="lg")
                status = gr.Markdown()
                gr.Markdown(HELP)
            with gr.Column(scale=2):
                with gr.Tabs():
                    with gr.Tab("🎬 Brain movie"):
                        movie_out = gr.Video(label="Reel + predicted brain response", height=560)
                    with gr.Tab("🧊 3D brain"):
                        gr.Markdown("Drag to rotate, scroll to zoom, use the slider or ▶ Play to move "
                                    "through time, and the buttons at the top for preset views.")
                        plot3d = gr.HTML()
                    with gr.Tab("📈 Intensity"):
                        intensity = gr.Plot(label="Intensity over time")
                        table = gr.Dataframe(label="Per-network summary", interactive=False, wrap=True)
                    with gr.Tab("⭐ Key moments"):
                        avg_img = gr.Image(label="Average activation", type="filepath")
                        gallery = gr.Gallery(label="Peak moments", columns=1, height="auto")
                    with gr.Tab("⬇️ Downloads"):
                        files = gr.File(label="Results", file_count="multiple")
        run_btn.click(
            analyze,
            inputs=[video_in, image_in, image_seconds, clip_start, clip_seconds, language, colormap, threshold, scale_max, make_movie],
            outputs=[status, movie_out, plot3d, intensity, avg_img, gallery, table, files],
        )
    return demo


def main():
    global DEMO
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true", help="synthetic predictions (no GPU/weights)")
    ap.add_argument("--share", action="store_true", help="create a public gradio.live link")
    ap.add_argument("--port", type=int, default=7860)
    ap.add_argument("--preload", action="store_true", help="load the model before serving")
    ap.add_argument("--no-warmup", action="store_true", help="skip background model download")
    args = ap.parse_args()
    DEMO = args.demo
    CACHE.mkdir(parents=True, exist_ok=True)
    if args.preload and not DEMO:
        get_model()
    elif not DEMO and not args.no_warmup:
        import threading

        threading.Thread(target=warm_up, daemon=True).start()
    build_ui().queue(max_size=8).launch(
        server_name="0.0.0.0", server_port=args.port, share=args.share,
        allowed_paths=[str(CACHE)], show_error=True,
    )


if __name__ == "__main__":
    main()

In [ ]:
#@title 3 · Launch the app → click the gradio.live link that appears below
!python app.py --share --preload